# DeferralX Colab Notebook

This notebook runs a faster, resume-safe DeferralX pipeline on Google Colab.

Run code cells in order. If collection is interrupted, rerun the collection cell with `--resume`.


In [ ]:
# 1) Clone repo
!git clone https://github.com/elom354/deferralX
%cd deferralX


In [ ]:
# 2) Install dependencies
!python -m pip install --upgrade pip
!pip install "numpy<2" torch transformers datasets sentencepiece accelerate


In [ ]:
# 2b) Runtime stability settings
import os
os.environ['KMP_USE_SHM'] = '0'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
print('Runtime env configured')


In [ ]:
# 3) Build real questions from MMLU with domain/profile heterogeneity
!PYTHONPATH=src python -m deferralx.prepare_questions \
  --dataset cais/mmlu \
  --subset all \
  --split test \
  --output data/mmlu_questions_600.csv \
  --question-col question \
  --choices-col choices \
  --answer-col answer \
  --answer-is-index \
  --domain general \
  --domain-mode mmlu_subject \
  --profile-mode cycle \
  --severe-mode by_domain \
  --limit 600

In [ ]:
# 4) Collect local HF logs (faster + resume-safe)
# Re-run this cell if Colab disconnects; --resume continues from last processed row.
!PYTHONPATH=src python -m deferralx.run collect-local-hf \
  --questions data/mmlu_questions_600.csv \
  --output data/real_llm_logs_local.csv \
  --audit-jsonl outputs/audit/real_collection_local_hf.jsonl \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --device auto \
  --max-tokens 96 \
  --agreement-samples 1 \
  --skip-confidence-pass \
  --resume


In [ ]:
# 5) Evaluate routing policies
!PYTHONPATH=src python -m deferralx.run run \
  --input data/real_llm_logs_local.csv \
  --outdir outputs/main_colab \
  --utility-config configs/utility_config.json \
  --test-ratio 0.30 \
  --seed 42 \
  --bootstrap 200


In [ ]:
# 6) View results
!cat outputs/main_colab/summary.md
!head -n 20 outputs/main_colab/metrics_overall.csv


In [ ]:
# 7) Optional: download artifacts
from google.colab import files
files.download('outputs/main_colab/summary.md')
files.download('outputs/main_colab/metrics_overall.csv')
